# 07 — Final Held-Out Evaluation & Leaderboard Comparison

**The official TDC test set is touched here, once, for every already-frozen model.**
No hyperparameter or architecture decisions are made after seeing these numbers — every
model below was selected/frozen using internal validation AUROC only (notebooks 03-06).

Each model is rebuilt from the config saved inside its own `experiments/<run>/metrics.json`
(not re-read from the `config/*.yaml` files, which may have been edited since training) --
this guarantees the architecture matches exactly what `best.ckpt` was actually trained with.


In [7]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from pathlib import Path

import torch
import pandas as pd
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGDataLoader

from src.utils import load_json, save_json
from src.featurizers import smiles_to_selfies, PAD
from src.datasets import SelfiesDataset, ConformerDataset, collate_conformers, MultimodalDataset, multimodal_collate
from src.graph_featurizer import mol_to_graph_data, ATOM_FEAT_DIM, BOND_FEAT_DIM
from src.models import Selfies1DModel, Graph2DModel, Conformer3DModel, MultimodalModel
from src.train_utils import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splits = load_json("../data/splits.json")
print("Test set size:", len(splits["test"]["id"]))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Test set size: 132


## Rebuild test sets for each modality

In [8]:
vocab = load_json("../data/processed/selfies_vocab.json")
selfies_test = [smiles_to_selfies(s) for s in splits["test"]["smiles"]]

test_selfies_ds = SelfiesDataset(selfies_test, splits["test"]["y"], vocab, max_len=128)
test_graph_ds = [mol_to_graph_data(smi, label) for smi, label in zip(splits["test"]["smiles"], splits["test"]["y"])]
test_conformer_ds = ConformerDataset(splits["test"]["id"], splits["test"]["y"], "../data/processed/conformers")
test_mm_ds = MultimodalDataset(
    ids=splits["test"]["id"], smiles_list=splits["test"]["smiles"], selfies_list=selfies_test,
    labels=splits["test"]["y"], vocab=vocab, max_len=128, conformers_dir="../data/processed/conformers",
)
print("all test sets built")


all test sets built


## Helper: rebuild a model from its saved metrics.json config, load best.ckpt, evaluate on test

In [9]:
def load_and_eval(experiment_dir, model_builder, forward_fn, loader, overrides=None):
    saved = load_json(Path(experiment_dir) / "metrics.json")
    if overrides:
        for section, updates in overrides.items():
            saved["config"][section].update(updates)
    model = model_builder(saved["config"])
    model.load_state_dict(torch.load(Path(experiment_dir) / "best.ckpt", map_location=device))
    model.to(device)
    test_metrics = evaluate(model, loader, forward_fn, device)
    return {
        "model": saved["model"],
        "val_auroc": saved["auroc"],
        "test_auroc": test_metrics["auroc"],
    }

results = []

### 1D-only

In [10]:
def build_1d(cfg):
    return Selfies1DModel(
        vocab_size=len(vocab), pad_id=vocab[PAD],
        d_model=cfg["model"]["d_model"], n_heads=cfg["model"]["n_heads"], n_layers=cfg["model"]["n_layers"],
        d_ff=cfg["model"]["d_ff"], dropout=cfg["model"]["dropout"], max_len=cfg["model"]["max_len"],
    )

def fwd_1d(model, batch, device):
    return model(batch["input_ids"].to(device))

loader_1d = DataLoader(test_selfies_ds, batch_size=32, shuffle=False)
results.append(load_and_eval("../experiments/1d_only", build_1d, fwd_1d, loader_1d))
results[-1]


{'model': '1d_only', 'val_auroc': 0.84625, 'test_auroc': 0.7592047128129602}

### 2D-only

In [11]:
def build_2d(cfg):
    return Graph2DModel(
        atom_feat_dim=ATOM_FEAT_DIM, bond_feat_dim=BOND_FEAT_DIM,
        hidden_dim=cfg["model"]["hidden_dim"], n_layers=cfg["model"]["n_layers"], dropout=cfg["model"]["dropout"],
    )

def fwd_2d(model, batch, device):
    batch = batch.to(device)
    return model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)

loader_2d = PyGDataLoader(test_graph_ds, batch_size=32, shuffle=False)
results.append(load_and_eval("../experiments/2d_only", build_2d, fwd_2d, loader_2d))
results[-1]


{'model': '2d_only', 'val_auroc': 0.88125, 'test_auroc': 0.7147275405007364}

### 3D — all three aggregation variants

In [12]:
def build_3d(cfg):
    return Conformer3DModel(
        agg_mode=cfg["model"]["agg_mode"],
        hidden_channels=cfg["model"]["hidden_channels"], num_filters=cfg["model"]["num_filters"],
        num_interactions=cfg["model"]["num_interactions"], num_gaussians=cfg["model"]["num_gaussians"],
        cutoff=cfg["model"]["cutoff"], max_z=cfg["model"]["max_z"],
    )

def fwd_3d(model, batch, device):
    return model(
        batch["atom_z"].to(device), batch["atom_pos"].to(device),
        batch["atom_conf_batch"].to(device), batch["conf_mol_batch"].to(device),
        batch["conf_weights"].to(device), num_mols=batch["num_mols"],
    )

loader_3d = DataLoader(test_conformer_ds, batch_size=32, shuffle=False, collate_fn=collate_conformers)
agg_modes = {
    "../experiments/3d_uniform_mean": "uniform_mean",
    "../experiments/3d_only": "weighted_mean",
    "../experiments/3d_learned_attention": "learned_attention",
}
import gc

for exp_dir, agg_mode in agg_modes.items():
    results.append(load_and_eval(exp_dir, build_3d, fwd_3d, loader_3d, overrides={"model": {"agg_mode": agg_mode}}))
    print(results[-1])
    gc.collect()
    torch.cuda.empty_cache()


{'model': '3d_uniform_mean', 'val_auroc': 0.848125, 'test_auroc': 0.8247422680412371}
{'model': '3d_weighted_mean', 'val_auroc': 0.825, 'test_auroc': 0.8241531664212076}
{'model': '3d_learned_attention', 'val_auroc': 0.88625, 'test_auroc': 0.8350515463917526}


### Multimodal — concat and gated

In [13]:
def build_multimodal(cfg):
    return MultimodalModel(
        vocab_size=len(vocab), pad_id=vocab[PAD],
        selfies_kwargs=dict(cfg["encoders"]["selfies"]),
        graph_kwargs={"atom_feat_dim": ATOM_FEAT_DIM, "bond_feat_dim": BOND_FEAT_DIM, **cfg["encoders"]["graph"]},
        schnet_kwargs=dict(cfg["encoders"]["schnet"]),
        fusion_type=cfg["fusion"]["type"] if "type" in cfg["fusion"] else cfg["fusion"].get("type"),
        fusion_dim=cfg["fusion"]["fusion_dim"],
    )

def fwd_mm(model, batch, device):
    graph = batch["graph"].to(device)
    return model(
        batch["input_ids"].to(device),
        graph.x, graph.edge_index, graph.edge_attr, graph.batch,
        batch["conf_atom_z"].to(device), batch["conf_atom_pos"].to(device),
        batch["conf_atom_conf_batch"].to(device), batch["conf_conf_mol_batch"].to(device),
        batch["conf_conf_weights"].to(device), batch["conf_num_mols"],
    )

loader_mm = DataLoader(test_mm_ds, batch_size=32, shuffle=False, collate_fn=multimodal_collate)

for exp_dir, fusion_type in [("../experiments/multimodal_concat", "concat"), ("../experiments/multimodal_fusion", "gated")]:
    results.append(load_and_eval(exp_dir, build_multimodal, fwd_mm, loader_mm, overrides={"fusion": {"type": fusion_type}}))
    print(results[-1])


{'model': 'multimodal_concat', 'val_auroc': 0.88375, 'test_auroc': 0.8132547864506627}
{'model': 'multimodal_gated', 'val_auroc': 0.84625, 'test_auroc': 0.7425625920471282}


## Final comparison table

This is the headline table for the report.

In [14]:
final_table = pd.DataFrame(results).sort_values("test_auroc", ascending=False).reset_index(drop=True)
final_table


,model,val_auroc,test_auroc
0,3d_learned_attention,0.886250,0.835052
1,3d_uniform_mean,0.848125,0.824742
2,3d_weighted_mean,0.825000,0.824153
3,multimodal_concat,0.883750,0.813255
4,1d_only,0.846250,0.759205
5,multimodal_gated,0.846250,0.742563
6,2d_only,0.881250,0.714728


## Comparison against the TDC hERG leaderboard

Leaderboard reference (fetched from the official TDC hERG leaderboard page; values are
mean +/- std across TDC's 5 official scaffold-split seeds.

| Rank | Model | AUROC |
|---|---|---|
| 1 | MapLight + GNN | 0.880 +/- 0.002 |
| 2 | CFA | 0.875 +/- 0.014 |
| 4 | MapLight | 0.871 +/- 0.004 |
| 6 | MiniMol | 0.846 +/- 0.016 |
| 10 | AttentiveFP (GNN baseline) | 0.825 +/- 0.007 |
| 16 | GCN (GNN baseline) | 0.738 +/- 0.038 |



In [15]:
save_json(final_table.to_dict(orient="records"), "../experiments/final_comparison_table.json")
print("Saved final comparison table.")


Saved final comparison table.


In [16]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix,
)

@torch.no_grad()
def full_classification_metrics(model, loader, forward_fn, device, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        logits = forward_fn(model, batch, device)
        all_logits.append(logits.detach().cpu().numpy())
        all_labels.append(batch["label"].detach().cpu().numpy())

    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= threshold).astype(int)

    return {
        "auroc": roc_auc_score(labels, probs),
        "auprc": average_precision_score(labels, probs),
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "mcc": matthews_corrcoef(labels, preds),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
    }


def load_model_for_metrics(experiment_dir, model_builder, overrides=None):
    saved = load_json(Path(experiment_dir) / "metrics.json")
    if overrides:
        for section, updates in overrides.items():
            saved["config"][section].update(updates)
    model = model_builder(saved["config"])
    model.load_state_dict(torch.load(Path(experiment_dir) / "best.ckpt", map_location=device))
    model.to(device)
    return model


metrics_specs = [
    ("1d_only",              "../experiments/1d_only",              build_1d,          fwd_1d, loader_1d, None),
    ("2d_only",               "../experiments/2d_only",               build_2d,          fwd_2d, loader_2d, None),
    ("3d_uniform_mean",       "../experiments/3d_uniform_mean",       build_3d,          fwd_3d, loader_3d, {"model": {"agg_mode": "uniform_mean"}}),
    ("3d_weighted_mean",      "../experiments/3d_only",               build_3d,          fwd_3d, loader_3d, {"model": {"agg_mode": "weighted_mean"}}),
    ("3d_learned_attention",  "../experiments/3d_learned_attention",  build_3d,          fwd_3d, loader_3d, {"model": {"agg_mode": "learned_attention"}}),
    ("multimodal_concat",    "../experiments/multimodal_concat",     build_multimodal,  fwd_mm, loader_mm, {"fusion": {"type": "concat"}}),
    ("multimodal_gated",     "../experiments/multimodal_fusion",     build_multimodal,  fwd_mm, loader_mm, {"fusion": {"type": "gated"}}),
]

all_metrics = []
for name, exp_dir, builder, forward_fn, loader, overrides in metrics_specs:
    model = load_model_for_metrics(exp_dir, builder, overrides)
    m = full_classification_metrics(model, loader, forward_fn, device)
    print(f"{name:22s}  AUROC={m['auroc']:.4f}  AUPRC={m['auprc']:.4f}  F1={m['f1']:.4f}  MCC={m['mcc']:.4f}")
    all_metrics.append({"model": name, **{k: v for k, v in m.items() if k != "confusion_matrix"}})

metrics_table = pd.DataFrame(all_metrics).sort_values("auroc", ascending=False).reset_index(drop=True)
save_json(all_metrics, "../experiments/final_full_metrics.json")
metrics_table

1d_only                 AUROC=0.7592  AUPRC=0.8863  F1=0.8558  MCC=0.2947
2d_only                 AUROC=0.7147  AUPRC=0.8524  F1=0.8626  MCC=0.3614
3d_uniform_mean         AUROC=0.8252  AUPRC=0.9179  F1=0.8716  MCC=0.3778
3d_weighted_mean        AUROC=0.8243  AUPRC=0.9120  F1=0.8651  MCC=0.3505
3d_learned_attention    AUROC=0.8351  AUPRC=0.9374  F1=0.8638  MCC=0.3553
multimodal_concat       AUROC=0.8133  AUPRC=0.9109  F1=0.8837  MCC=0.4619
multimodal_gated        AUROC=0.7426  AUPRC=0.8671  F1=0.8585  MCC=0.3843


,model,auroc,auprc,accuracy,precision,recall,f1,mcc
0,3d_learned_attention,0.835052,0.937403,0.780303,0.793103,0.948454,0.863850,0.355350
1,3d_uniform_mean,0.825184,0.917888,0.787879,0.785124,0.979381,0.871560,0.377752
2,3d_weighted_mean,0.824300,0.911983,0.780303,0.788136,0.958763,0.865116,0.350472
3,multimodal_concat,0.813255,0.910917,0.810606,0.805085,0.979381,0.883721,0.461947
4,1d_only,0.759205,0.886297,0.765152,0.779661,0.948454,0.855814,0.294734
5,multimodal_gated,0.742563,0.867051,0.780303,0.814815,0.907216,0.858537,0.384297
6,2d_only,0.714728,0.852408,0.780303,0.798246,0.938144,0.862559,0.361443
